In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
pd.set_option("display.max_column",500)
pd.set_option("display.max_row",500)

In [3]:
# To read the second sheet (index 1)
private_18_19 = pd.read_excel('organized_education_data/private_schools_2018-2019.xlsx', sheet_name="Schools, Classes & Staff by Pro")
private_19_20 = pd.read_excel('organized_education_data/private_schools_2019-2020.xlsx', sheet_name="Schools, Classes & Staff by Pro")
private_20_21 = pd.read_excel('organized_education_data/private_schools_2020-2021.xlsx', sheet_name="Schools, Classes & Staff by Pro")

In [4]:
row =private_18_19.iloc[0]
private_18_19 = private_18_19.rename(columns=row)
private_18_19 = private_18_19.iloc[2:-3]
private_18_19 = private_18_19.reset_index(drop=True)
private_18_19 = private_18_19.set_index("Province")
cols = list(private_18_19.columns)
cols[3] = "Enrollment(F)"
cols[5] = "Teaching Staff(F)"
cols[7] = "Non-Teaching Staff(F)"
cols[10] = "Foreigner Teaching Staff(F)"
private_18_19.columns = cols
private_18_19 = private_18_19.astype('Int64')

In [5]:
private_18_19.shape

(25, 14)

In [6]:
private_19_20 =private_19_20.iloc[1:-3]
private_19_20 = private_19_20.reset_index(drop=True)
private_19_20 = private_19_20.set_index("Province")
cols = list(private_19_20.columns)
cols[3] = "Enrollment(F)"
cols[5] = "Teaching Staff(F)"
cols[7] = "Non-Teaching Staff(F)"
cols[10] = "Foreigner Teaching Staff(F)"
private_19_20.columns = cols
private_19_20 = private_19_20.astype('Int64')

In [7]:
row =private_20_21.iloc[0]
private_20_21 = private_20_21.rename(columns=row)
private_20_21 = private_20_21.iloc[2:-3]
private_20_21 = private_20_21.reset_index(drop=True)
private_20_21 = private_20_21.set_index("Province")
cols = list(private_20_21.columns)
cols[3] = "Enrollment(F)"
cols[5] = "Teaching Staff(F)"
cols[7] = "Non-Teaching Staff(F)"
cols[9] = "Foreigner Teaching Staff(F)"
private_20_21.columns = cols
private_20_21 = private_20_21.astype('Int64')
private_20_21.shape

(25, 13)

In [8]:
private_18_19.insert(0,'Year',2018)
private_19_20.insert(0,'Year',2019)
private_20_21.insert(0,'Year',2020)

In [9]:
combined_df = pd.concat([private_18_19, private_19_20, private_20_21])

In [10]:
combined_df.columns

Index(['Year', 'Number of\nSchools', 'Number of\nClasses', 'Enrollment',
       'Enrollment(F)', 'Teaching Staff', 'Teaching Staff(F)',
       'Non-Teaching Staff', 'Non-Teaching Staff(F)', 'Govern. staff',
       'Foreigner Teaching Staff', 'Foreigner Teaching Staff(F)', 'Buildings',
       'Rooms', 'Classrooms'],
      dtype='object')

In [11]:
# 1. Define the mapping dictionary
rename_map = {
    'year': 'Year',
    'Number of\nSchools': 'Schools',
    'Number of\nClasses': 'Classes',
    'Enrollment': 'Enrollment_Total',
    'Enrollment(F)': 'Enrollment_Girl',
    'Teaching Staff': 'TeachStaff_Total',
    'Teaching Staff(F)': 'TeachStaff_Female',
    'Non-Teaching Staff': 'NonTeachStaff_Total',
    'Non-Teaching Staff(F)': 'NonTeachStaff_Female',
    'Govern. staff': 'Govern_Staff',
    'Foreigner Teaching Staff': 'Foreigner_Staff_Total',
    'Foreigner Teaching Staff(F)': 'Foreigner_Staff_Female',
    'Buildings': 'Buildings',
    'Rooms': 'Rooms',
    'Classrooms': 'Classrooms'
}

# 2. Apply the rename to your combined dataframe
# errors='ignore' ensures it won't crash if a column is already renamed
combined_df = combined_df.rename(columns=rename_map, errors='ignore')

# 3. Double check the result
print(combined_df.columns.tolist())

['Year', 'Schools', 'Classes', 'Enrollment_Total', 'Enrollment_Girl', 'TeachStaff_Total', 'TeachStaff_Female', 'NonTeachStaff_Total', 'NonTeachStaff_Female', 'Govern_Staff', 'Foreigner_Staff_Total', 'Foreigner_Staff_Female', 'Buildings', 'Rooms', 'Classrooms']


In [12]:
combined_df = combined_df.drop(columns=['Govern_Staff', 'Foreigner_Staff_Total', 'Foreigner_Staff_Female', 'Buildings', 'Enrollment_Girl','Rooms', 'Classrooms','Enrollment_Total'])

In [13]:
combined_df.columns

Index(['Year', 'Schools', 'Classes', 'TeachStaff_Total', 'TeachStaff_Female',
       'NonTeachStaff_Total', 'NonTeachStaff_Female'],
      dtype='object')

In [14]:
combined_df.shape

(75, 7)

In [15]:
def set_province_index(df):
    """Rename the first column to Province, drop junk rows, set as index."""
    df = df.rename(columns={df.columns[0]: "Province"})
    # Drop rows where Province is NaN (header-junk rows 0 and 1)
    df = df[df["Province"].notna()].copy()
    df = df.reset_index(drop=True)
    df = df.set_index("Province")
    return df


# ═══════════════════════════════════════════════════════════════
# 1.  ENROLLMENT & REPEATERS  (Enroll_and_repeat_18_19 / 19_20 / 20_21)
# ═══════════════════════════════════════════════════════════════

Enroll_and_repeat_18_19 = pd.read_excel(
    'organized_education_data/private_schools_2018-2019.xlsx',
    sheet_name="Enrollment & Repeaters by Grade"
)
Enroll_and_repeat_19_20 = pd.read_excel(
    'organized_education_data/private_schools_2019-2020.xlsx',
    sheet_name="Enrollment & Repeaters by Grade"
)
Enroll_and_repeat_20_21 = pd.read_excel(
    'organized_education_data/private_schools_2020-2021.xlsx',
    sheet_name="Enrollment & Repeaters by Grade"
)

# Build clean column names (same for all three years)
sub_headers = ['Enrollment_Total', 'Enrollment_Girl', 'Repeaters_Total', 'Repeaters_Girl']
enroll_cols = ['Province']
for i in range(1, 13):
    for sub in sub_headers:
        enroll_cols.append(f'Grade_{i}_{sub}')
for sub in sub_headers:
    enroll_cols.append(f'Total_{sub}')


def clean_enroll(df):
    df.columns = enroll_cols
    # 1. Drop the junk sub-header rows
    df = df[df["Province"].notna()].copy()
    
    # 2. REMOVE FOOTERS: Filter out aggregate rows
    # This is more robust than [:-3] because it looks for the names specifically
    df = df[~df["Province"].astype(str).str.contains("Whole Kingdom|Area", case=False, na=False)]
    
    df = df.reset_index(drop=True)
    df = df.set_index("Province")
    
    df = df.apply(pd.to_numeric, errors='coerce').astype('Int64')
    return df


Enroll_and_repeat_18_19 = clean_enroll(Enroll_and_repeat_18_19)
Enroll_and_repeat_19_20 = clean_enroll(Enroll_and_repeat_19_20)
Enroll_and_repeat_20_21 = clean_enroll(Enroll_and_repeat_20_21)

# Add Year and combine
Enroll_and_repeat_18_19['Year'] = 2018
Enroll_and_repeat_19_20['Year'] = 2019
Enroll_and_repeat_20_21['Year'] = 2020

combined_enroll = pd.concat(
    [Enroll_and_repeat_18_19, Enroll_and_repeat_19_20, Enroll_and_repeat_20_21]
)
# Move Year to front
cols = list(combined_enroll.columns)
cols.insert(0, cols.pop(cols.index('Year')))
combined_enroll = combined_enroll[cols]


# ═══════════════════════════════════════════════════════════════
# 2.  TEACHING STAFF BY EDUCATION LEVEL (Staff_18_19 / 19_20 / 20_21)
# ═══════════════════════════════════════════════════════════════

Staff_18_19 = pd.read_excel(
    'organized_education_data/private_schools_2018-2019.xlsx',
    sheet_name="Teaching Staff by Education Lev"
)
Staff_19_20 = pd.read_excel(
    'organized_education_data/private_schools_2019-2020.xlsx',
    sheet_name="Teaching Staff by Education Lev"
)
Staff_20_21 = pd.read_excel(
    'organized_education_data/private_schools_2020-2021.xlsx',
    sheet_name="Teaching Staff by Education Lev"
)

# Column structure (19 columns):
# Province | Total: Primary L.sec U.Sec Graduate PostGrad PhD
#           | Female: Primary L.sec U.Sec Graduate PostGrad PhD
#           | Without Ped Training: Primary L.sec U.Sec Graduate PostGrad PhD
staff_cols = ['Province',
    'Total_Primary', 'Total_LSec', 'Total_USec', 'Total_Graduate', 'Total_PostGrad', 'Total_PhD',
    'Female_Primary', 'Female_LSec', 'Female_USec', 'Female_Graduate', 'Female_PostGrad', 'Female_PhD',
    'NoPedTrain_Primary', 'NoPedTrain_LSec', 'NoPedTrain_USec',
    'NoPedTrain_Graduate', 'NoPedTrain_PostGrad', 'NoPedTrain_PhD'
]


def clean_staff(df):
    # Skip row 0 (headers) and skip the last 3 rows (footers)
    df = df.iloc[1:-3].copy() 
    
    df.columns = staff_cols
    df = df[df["Province"].notna()].copy()
    df = df.set_index("Province")
    
    # Optional: double check to remove any remaining "Total" rows
    df = df[~df.index.astype(str).str.contains("Total|Whole", case=False, na=False)]
    
    df = df.apply(pd.to_numeric, errors='coerce').astype('Int64')
    return df


Staff_18_19 = clean_staff(Staff_18_19)
Staff_19_20 = clean_staff(Staff_19_20)
Staff_20_21 = clean_staff(Staff_20_21)

# Add Year and combine
Staff_18_19['Year'] = 2018
Staff_19_20['Year'] = 2019
Staff_20_21['Year'] = 2020

combined_staff = pd.concat([Staff_18_19, Staff_19_20, Staff_20_21])
cols = list(combined_staff.columns)
cols.insert(0, cols.pop(cols.index('Year')))
combined_staff = combined_staff[cols]

In [16]:
combined_staff.columns

Index(['Year', 'Total_Primary', 'Total_LSec', 'Total_USec', 'Total_Graduate',
       'Total_PostGrad', 'Total_PhD', 'Female_Primary', 'Female_LSec',
       'Female_USec', 'Female_Graduate', 'Female_PostGrad', 'Female_PhD',
       'NoPedTrain_Primary', 'NoPedTrain_LSec', 'NoPedTrain_USec',
       'NoPedTrain_Graduate', 'NoPedTrain_PostGrad', 'NoPedTrain_PhD'],
      dtype='object')

In [17]:
combined_staff = combined_staff.drop(columns=[ 'Female_Primary', 'Female_LSec',
       'Female_USec', 'Female_Graduate', 'Female_PostGrad', 'Female_PhD',
       'NoPedTrain_Primary', 'NoPedTrain_LSec', 'NoPedTrain_USec',
       'NoPedTrain_Graduate', 'NoPedTrain_PostGrad', 'NoPedTrain_PhD'])

In [18]:
rename_map = {
    "Total_Primary": "T_Primary",
    "Total_LSec": "T_LSec",
    "Total_USec": "T_USec",
    "Total_Graduate": "T_Graduate",
    "Total_PostGrad": "T_PostGrad", # Check if this logic holds for your data!
    "Total_PhD": "T_PhD",
}

# 1. Rename the columns
combined_staff = combined_staff.rename(columns=rename_map)

In [19]:
private_GER_18_19 = pd.read_excel('organized_education_data/private_schools_2018-2019.xlsx', sheet_name="Access & Admission Rate Indicat")
private_GER_19_20 = pd.read_excel('organized_education_data/private_schools_2019-2020.xlsx', sheet_name="Access & Admission Rate Indicat")
private_GER_20_21 = pd.read_excel('organized_education_data/private_schools_2020-2021.xlsx', sheet_name="Access & Admission Rate Indicat")

In [20]:
cols = ["Province","GAR(Total)","GAR(F)","GAR(M)","NAR(Total)","NAR(F)","NAR(M)","GER pri","GER L.Sec","GER U.Sec","GER pri(F)","GER L.Sec(F)","GER U.Sec(F)","GER pri(M)","GER L.Sec(M)","GER U.Sec(M)","NER pri","NER L.Sec","NER U.Sec","NER pri(F)","NER L.Sec(F)","NER U.Sec(F)","NER pri(M)","NER L.Sec(M)","NER U.Sec(M)"]
private_GER_18_19.columns = cols
private_GER_18_19 = private_GER_18_19[2:-3]
private_GER_18_19 = private_GER_18_19.set_index("Province")
private_GER_18_19 = private_GER_18_19.astype('float64')

In [21]:
private_GER_19_20.columns = cols
private_GER_19_20 = private_GER_19_20[2:-3]
private_GER_19_20 = private_GER_19_20.set_index("Province")
private_GER_19_20 = private_GER_19_20.astype('float64')

In [22]:
private_GER_20_21.columns = cols
private_GER_20_21 = private_GER_20_21[3:-3]
private_GER_20_21 = private_GER_20_21.set_index("Province")
private_GER_20_21 = private_GER_20_21.astype('float64')

In [23]:
private_GER_18_19.insert(0,'year',2018)
private_GER_19_20.insert(0,'year',2019)
private_GER_20_21.insert(0,'year',2020)

In [24]:
GER = pd.concat([private_GER_18_19,private_GER_19_20,private_GER_20_21])

In [25]:
GER.columns

Index(['year', 'GAR(Total)', 'GAR(F)', 'GAR(M)', 'NAR(Total)', 'NAR(F)',
       'NAR(M)', 'GER pri', 'GER L.Sec', 'GER U.Sec', 'GER pri(F)',
       'GER L.Sec(F)', 'GER U.Sec(F)', 'GER pri(M)', 'GER L.Sec(M)',
       'GER U.Sec(M)', 'NER pri', 'NER L.Sec', 'NER U.Sec', 'NER pri(F)',
       'NER L.Sec(F)', 'NER U.Sec(F)', 'NER pri(M)', 'NER L.Sec(M)',
       'NER U.Sec(M)'],
      dtype='object')

In [26]:
private_COM_18_19 = pd.read_excel('organized_education_data/private_schools_2018-2019.xlsx', sheet_name="Completion Rate 2018-2019")
private_COM_19_20 = pd.read_excel('organized_education_data/private_schools_2019-2020.xlsx', sheet_name="Completion Rate 2019-2020")
private_COM_20_21 = pd.read_excel('organized_education_data/private_schools_2020-2021.xlsx', sheet_name="Completion Rate 2021-2022")

In [27]:
com_col = ['Province','pri(Total)','pri(F)','pri(M)','L.sec(Total)','L.sec(F)','L.sec(M)','U.sec(Total)','U.sec(F)','U.sec(M)']
private_COM_18_19.columns = com_col
private_COM_18_19 = private_COM_18_19[1:-3]
private_COM_18_19 = private_COM_18_19.set_index("Province")
private_COM_18_19 = private_COM_18_19.astype('float64')

In [28]:
private_COM_19_20.columns = com_col
private_COM_19_20 = private_COM_19_20[1:-3]
private_COM_19_20 = private_COM_19_20.set_index("Province")
private_COM_19_20 = private_COM_19_20.astype('float64')

In [29]:
private_COM_20_21.columns = com_col
private_COM_20_21 = private_COM_20_21[2:-3]
private_COM_20_21 = private_COM_20_21.set_index("Province")
private_COM_20_21 = private_COM_20_21.astype('float64')

In [30]:
private_COM_18_19.insert(0,'year',2018)
private_COM_19_20.insert(0,'year',2019)
private_COM_20_21.insert(0,'year',2020)

In [31]:
COM = pd.concat([private_COM_18_19,private_COM_19_20,private_COM_20_21])

In [32]:
COM.columns

Index(['year', 'pri(Total)', 'pri(F)', 'pri(M)', 'L.sec(Total)', 'L.sec(F)',
       'L.sec(M)', 'U.sec(Total)', 'U.sec(F)', 'U.sec(M)'],
      dtype='object')

In [33]:
private_ENR_18_19 = pd.read_excel('organized_education_data/private_schools_2018-2019.xlsx', sheet_name="Enrollment by Level 2018-2019")
private_ENR_19_20 = pd.read_excel('organized_education_data/private_schools_2019-2020.xlsx', sheet_name="Enrollment by Level 2019-2020")
private_ENR_20_21 = pd.read_excel('organized_education_data/private_schools_2020-2021.xlsx', sheet_name="Enrollment by Level 2020-2021")

In [34]:
# 1. Include 'Province' at the start (Total of 14 columns)
ENR_COL = [
    'Province', 'Number of school', 'pre(Total)', 'pre(F)', 
    'pri(Total)', 'pri(F)', 'L.sec(Total)', 'L.sec(F)', 
    'U.sec(Total)', 'U.sec(F)', 'Total(Total)', 'Total(F)', 
    'Female pri(%)', 'Female all(%)'
]
private_ENR_18_19.columns = ENR_COL
private_ENR_18_19 = private_ENR_18_19.iloc[1:-3].copy()
private_ENR_18_19 = private_ENR_18_19.set_index("Province")
private_ENR_18_19 = private_ENR_18_19.apply(pd.to_numeric, errors='coerce')

In [35]:
private_ENR_19_20.columns = ENR_COL
private_ENR_19_20 = private_ENR_19_20.iloc[1:-3].copy()
private_ENR_19_20 = private_ENR_19_20.set_index("Province")
private_ENR_19_20 = private_ENR_19_20.apply(pd.to_numeric, errors='coerce')

In [36]:
private_ENR_20_21.columns = ENR_COL
private_ENR_20_21 = private_ENR_20_21.iloc[1:-3].copy()
private_ENR_20_21 = private_ENR_20_21.set_index("Province")
private_ENR_20_21 = private_ENR_20_21.apply(pd.to_numeric, errors='coerce')

In [37]:
private_ENR_18_19.insert(0,'year',2018)
private_ENR_19_20.insert(0,'year',2019)
private_ENR_20_21.insert(0,'year',2020)

In [38]:
ENR = pd.concat([private_ENR_18_19, private_ENR_19_20, private_ENR_20_21])

In [39]:
COM.columns

Index(['year', 'pri(Total)', 'pri(F)', 'pri(M)', 'L.sec(Total)', 'L.sec(F)',
       'L.sec(M)', 'U.sec(Total)', 'U.sec(F)', 'U.sec(M)'],
      dtype='object')

In [40]:
private_flow = pd.read_csv('public_school_flow_rates.csv')

In [41]:

combined_df = combined_df.reset_index().set_index(['Province', 'Year'])
combined_enroll = combined_enroll.reset_index().set_index(['Province', 'Year'])
combined_staff = combined_staff.reset_index().set_index(['Province', 'Year'])
private_flow = private_flow.reset_index().set_index(['Province', 'Year'])

# Now try the concat again
private_school_2018_2020 = pd.concat([
    combined_df, 
    combined_enroll, 
    combined_staff,
    private_flow
], axis=1)

In [42]:
cols = private_school_2018_2020.columns.tolist()

# Print 5 columns per line
for i in range(0, len(cols), 5):
    print(f"{i:3}: {', '.join(cols[i:i+5])}")

  0: Schools, Classes, TeachStaff_Total, TeachStaff_Female, NonTeachStaff_Total
  5: NonTeachStaff_Female, Grade_1_Enrollment_Total, Grade_1_Enrollment_Girl, Grade_1_Repeaters_Total, Grade_1_Repeaters_Girl
 10: Grade_2_Enrollment_Total, Grade_2_Enrollment_Girl, Grade_2_Repeaters_Total, Grade_2_Repeaters_Girl, Grade_3_Enrollment_Total
 15: Grade_3_Enrollment_Girl, Grade_3_Repeaters_Total, Grade_3_Repeaters_Girl, Grade_4_Enrollment_Total, Grade_4_Enrollment_Girl
 20: Grade_4_Repeaters_Total, Grade_4_Repeaters_Girl, Grade_5_Enrollment_Total, Grade_5_Enrollment_Girl, Grade_5_Repeaters_Total
 25: Grade_5_Repeaters_Girl, Grade_6_Enrollment_Total, Grade_6_Enrollment_Girl, Grade_6_Repeaters_Total, Grade_6_Repeaters_Girl
 30: Grade_7_Enrollment_Total, Grade_7_Enrollment_Girl, Grade_7_Repeaters_Total, Grade_7_Repeaters_Girl, Grade_8_Enrollment_Total
 35: Grade_8_Enrollment_Girl, Grade_8_Repeaters_Total, Grade_8_Repeaters_Girl, Grade_9_Enrollment_Total, Grade_9_Enrollment_Girl
 40: Grade_9_Repeat

In [43]:
private_school_2018_2020.to_csv('private_cleaned.csv')